# OSR-303 Voltage Short-Circuit Fault Injection Example

## Connection Diagram

Use [powershorter](https://gitee.com/osr-tech/powershorter) for short-circuit faults and reset control, and a PICO3206D to observe the glitch.

![image](images/fault-example.png)

## Download the Project

As shown in `https://gitee.com/osr-tech/osr-303/blob/master/project/STM32CubeIDE-303Guide.md`, we use STM32CubeIDE to download the `FORLOOP` project (in the project folder).

## Communication Test

Since we have not yet configured the `GPIO` of `powershorter`, it outputs a low level by default, and the 303's reset signal is active-low. At this point, you can first disconnect the reset interface to perform a communication test.

In [1]:
import serial

In [2]:
loopv = 200

In [3]:
toe = serial.Serial('/dev/cu.usbmodem21302', 115200, timeout=1)

In [49]:
toe.write(loopv.to_bytes(1, 'little'))
ret = toe.read(1)
retv = int.from_bytes(ret, 'little')
print(retv)

200


## Controlling the 303 Reset

Use the `GPIO` output of `powershorter` to drive the 303's reset interface and reset the 303.

In [5]:
import power_shorter as ps
import time

In [6]:
ps_dev = ps.PowerShorter('/dev/cu.usbserial-2140') # select the serial port

In [7]:
def reset_toe():
    ps_dev.gpio(ps.GPIO.GPIO1, 0)
    time.sleep(0.3)
    ps_dev.gpio(ps.GPIO.GPIO1, 1)

In [13]:
# Test the reset functionality
reset_toe()  
toe.write(loopv.to_bytes(1, 'little'))
ret = toe.read(1)
retv = int.from_bytes(ret, 'little')
print(retv)

200


In [61]:
ps_dev.engine_cfg?

Signature:
ps_dev.engine_cfg(
    engine: power_shorter.ctrl.Engine,
    pattern: list,
    trigger_mode: power_shorter.ctrl.TRIGGER_MODE = <TRIGGER_MODE.RISE: 0>,
    pattern_repeat: int = 1,
    trigger_edges: int = 1,
)
Docstring:
set trigger and attack pattern for power shorter engine

Args:
    engine (Engine): which engine to be set.
    pattern (list): list of attack pattern of (level, duration), at most 8 can be set. level=0 means MOSFET off, level=1 means MOSFET on. duration is in unit of 10ns.
        Note that level in the last pattern will last until next arm. 
        e.g. [(0, 12), (1, 10), (0, 100), (1, 23), (0, 1)] means set MOSFET to '0' for 12*10ns, '1' for 10*10ns, '0' for 100*10ns, '1' for 23*10ns, '0' for 1*10ns. 
    trigger_mode: (TRIGGER_MODE): trigger mode. Defaults to TRIGGER_MODE.RISE.
    pattern_repeat (int, optional): pattern repeats. Defaults to 1.
    trigger_edges (int, optional): how many edges as trigger event. Defaults to 1.
File:      ~/Gitlab/atlas

## Short-Circuit Fault Injection

In [14]:
def glitch(delay, pulse):
    ps_dev.engine_cfg(ps.Engine.E1, [(0, delay), (1, pulse), (0, 1)]) # on receiving the trigger signal, wait delay*10 ns, then short-circuit for pulse*10 ns
    ps_dev.arm(ps.Engine.E1)
    toe.write(loopv.to_bytes(1, 'little'))
    ret = toe.read(1)
    retv = int.from_bytes(ret, 'little')
    state = None
    if ret == b'':
        state = 'dead'
        reset_toe()
    elif retv == loopv:
        state = 'normal'
    else:
        state = 'glitch success'
    return state, retv, delay, pulse

In [24]:
glitch(3200, 10)

('normal', 200, 3200, 10)

The pico oscilloscope can be used to observe the generation of the short-circuit glitch.

![image](images/fault-example-pico.png)

## Visualizing Fault Parameters and Results

You can conveniently observe fault parameters and results using [FaultViz](https://gitee.com/osr-tech/faultviz).

In [25]:
import faultviz

In [26]:
faultviz.start_view_service()

# Starting io.deephaven.python.server.EmbeddedServer
deephaven.cacheDir=/Users/ping/Library/Caches/io.Deephaven-Data-Labs.deephaven
deephaven.configDir=/Users/ping/Library/Application Support/io.Deephaven-Data-Labs.deephaven
deephaven.dataDir=/Users/ping/Library/Application Support/io.Deephaven-Data-Labs.deephaven
# io.deephaven.internal.log.LoggerFactoryServiceLoaderImpl: searching for 'io.deephaven.internal.log.LoggerFactory'...
# io.deephaven.internal.log.LoggerFactoryServiceLoaderImpl: found 'io.deephaven.internal.log.LoggerFactorySlf4j'
Server started on port 12345


In [27]:
vt = faultviz.ViewWidget()

In [28]:
state, retv, delay, pulse = glitch(3200, 20)
vt.update(state=state, val=retv, delay=delay, pulse=pulse)

## Random-Parameter Injection

In [30]:
from tqdm.notebook import tnrange
import random

In [60]:
for i in tnrange(2000):
    delay=random.randint(2500, 3000)
    pulse=random.randint(30, 36)
    state, retv, delay, pulse = glitch(delay, pulse)
    vt.update(state=state, val=retv, delay=delay, pulse=pulse)

  0%|          | 0/2000 [00:00<?, ?it/s]

## Results

In [53]:
vt.show()

DeephavenWidget(height=150, iframe_url='http://localhost:12345/iframe/table/?name=_15ed7d0d_a043_4d5d_8116_03d…

DeephavenWidget(height=600, iframe_url='http://localhost:12345/iframe/table/?name=_4be603af_b4da_4ec0_9baa_a87…